# 01_bronze_ingestion

Reads the raw `vgsales.csv` file from ADLS Gen2 using the direct `abfss://` path pattern and writes it as a Delta table in the Bronze layer. Run `00_setup_and_config` first in the same cluster session.

In [0]:
%run ./00_setup_and_config

In [0]:
# ============================================================
# PREREQUISITE CHECK
# ============================================================
# This notebook expects the setup notebook to have already run
# and defined shared variables such as RAW_PATH.

required_vars = ['RAW_PATH']
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise Exception(
        f"Missing required variables: {missing_vars}. Run 00_setup_and_config first."
    )

RAW_FILE_PATH = f"{RAW_PATH}vgsales.csv"
print('Raw file path:', RAW_FILE_PATH)

In [0]:
# ============================================================
# OPTIONAL FILE CHECK
# ============================================================
# Confirms the raw folder contents before reading the CSV.

display(dbutils.fs.ls(RAW_PATH))

In [0]:
# ============================================================
# READ RAW CSV
# ============================================================
# Bronze should preserve raw data as much as possible.

df_raw = spark.read.csv(
    RAW_FILE_PATH,
    header=True,
    inferSchema=True
)

print(f'Raw row count: {df_raw.count()}')
print(f'Columns: {df_raw.columns}')
df_raw.printSchema()
display(df_raw.limit(5))

In [0]:
# ============================================================
# ADD BRONZE INGESTION METADATA
# ============================================================

from pyspark.sql.functions import current_timestamp, lit, input_file_name

df_bronze = (
    df_raw
    .withColumn('_ingested_at', current_timestamp())
    .withColumn('_source_file', lit('_metadata.file_path'))
    .withColumn('_layer', lit('bronze'))
)

display(df_bronze.limit(3))

In [0]:
# ============================================================
# CREATE SCHEMA IF NEEDED
# ============================================================

spark.sql('CREATE SCHEMA IF NOT EXISTS gaming_bronze')
print('Schema gaming_bronze is ready.')

In [0]:
# ============================================================
# WRITE BRONZE DELTA TABLE
# ============================================================

df_bronze.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('gaming_bronze.vg_sales_raw')

print('Bronze Delta table written successfully.')
spark.sql('SELECT COUNT(*) AS total_rows FROM gaming_bronze.vg_sales_raw').show()

In [0]:
# ============================================================
# OPTIONAL OPTIMIZATION
# ============================================================

# OPTIMIZE is a Databricks SQL command that tidies up the table's storage.
# Over time, writing data can create many small files.
# Too many small files can make queries slower.
# 
# OPTIMIZE combines those small files into fewer, larger files
# so Spark can read the table more efficiently.
#
# In simple terms:
# - same data
# - same rows
# - faster reads / better performance
#
# This step is optional for learning projects,
# but it is a good habit in real data engineering workflows.

spark.sql('OPTIMIZE gaming_bronze.vg_sales_raw')

# Print a message so you know the command finished successfully.
print('OPTIMIZE complete.')

In [0]:
# ============================================================
# QUICK VALIDATION QUERY
# ============================================================

display(spark.sql(
    'SELECT Platform, Genre, COUNT(*) AS titles ' \
    'FROM gaming_bronze.vg_sales_raw ' \
    'GROUP BY Platform, Genre ORDER BY titles DESC LIMIT 20'
))